In [35]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import numpy as np

In [36]:
plots_dir = r"D:\PHD\Development\Python\Cardio_CAD\dash_viewer\classification\machine_learning\utils\plots"
os.makedirs(plots_dir, exist_ok=True)

In [37]:
def create_dataframe(classifier, data):
    df = pd.DataFrame(data)
    df.insert(0, "Classifier", classifier)
    return df

In [38]:
df_rf = create_dataframe("RF", {
    "Input": ["59 PC", "59 PC + GLS", " 59 PC + Echo"],
    "MCC": [0.3249, 0.4004, 0.4051],
    "Precision": [0.774, 0.8113, 0.8088],
    "Recall": [0.6933, 0.7741, 0.7231],
    "F1 Score": [0.7072, 0.7753, 0.7427],
    "Accuracy": [0.6933 , 0.6391 ,0.7231 ],
    "AUC": [0.6155 , 0.6391 , 0.674 ],
    "PR-AUC": [0.4116 , 0.4704 , 0.4931 ]
})


df_lr = create_dataframe("LR", {
    "Input": ["59 PC", "59 PC + GLS", " 59 PC + Echo"],
    "MCC": [0.2842, 0.2821, 0.4056],
    "Precision": [0.7777, 0.7864, 0.8148],
    "Recall": [0.6867, 0.7005, 0.7463],
    "F1 Score": [0.6883, 0.7041, 0.7453],
    "Accuracy": [0.6867, 0.7005, 0.7463],
    "AUC": [0.562, 0.5513, 0.6138],
    "PR-AUC": [0.3895, 0.3087, 0.4271]
})

df_xgb = create_dataframe("XGB", {
    "Input": ["59 PC", "59 PC + GLS", " 59 PC + Echo"],
    "MCC": [0.3584, 0.3963, 0.435],
    "Precision": [0.7967, 0.8267, 0.8199],
    "Recall": [0.66, 0.7003, 0.7882],
    "F1 Score": [0.6815, 0.6928, 0.7849],
    "Accuracy": [0.66, 0.7003, 0.7882],
    "AUC": [0.6461, 0.6054, 0.6426],
    "PR-AUC": [0.4157, 0.3737, 0.4841]
})


In [59]:
df_rf = create_dataframe("RF", {
    "Input": ["480 PC", "480 PC + GLS", "480 PC + Echo"],
    "MCC": [0.3276, 0.3729, 0.4618],
    "Precision": [0.7877, 0.8014, 0.8273],
    "Recall": [0.6667, 0.7452, 0.7596],
    "F1 Score": [0.6718, 0.754, 0.7727],
    "Accuracy": [0.6667, 0.7452, 0.7596],
    "AUC": [0.5973, 0.6239, 0.718],
    "PR-AUC": [0.4335, 0.4449, 0.4882]
})

df_xgb = create_dataframe("XGB", {
    "Input": ["480 PC", "480 PC + GLS", "480 PC + Echo"],
    "MCC": [0.3622, 0.3856, 0.3453],
    "Precision": [0.8304, 0.8153, 0.8388],
    "Recall": [0.58, 0.7077, 0.5682],
    "F1 Score": [0.5931, 0.7178, 0.5667],
    "Accuracy": [0.58, 0.7077, 0.5682],
    "AUC": [0.6273, 0.6468, 0.5614],
    "PR-AUC": [0.4073, 0.4637, 0.3631]
})

df_lr = create_dataframe("LR", {
    "Input": ["480 PC", "480 PC + GLS", "480 PC + Echo"],
    "MCC": [0.3828, 0.3015, 0.3241],
    "Precision": [0.8108, 0.7961, 0.7946],
    "Recall": [0.7333, 0.7296, 0.6663],
    "F1 Score": [0.7274, 0.7126, 0.6773],
    "Accuracy": [0.7333, 0.7296, 0.6663],
    "AUC": [0.5706, 0.505, 0.5929],
    "PR-AUC": [0.4351, 0.3446, 0.4107]
})


In [60]:
def plot_metrics(df, title, filename):
    df_melted = df.melt(id_vars=["Input"], var_name="Metric", value_name="Value")
    fig = px.bar(df_melted, x="Metric", y="Value", color="Input", barmode="group", title=title)
    fig.write_image(os.path.join(plots_dir, filename.replace(".html", ".png")))

In [65]:
plot_metrics(df_rf, "Random Forest Classifier Performance", "rf_performance.html")

In [66]:
plot_metrics(df_xgb, "XGBoost Classifier Performance", "xgb_performance.html")

In [67]:
plot_metrics(df_lr, "Logistic Regression Classifier Performance", "lr_performance.html")

In [64]:

df_combined = pd.concat([df_rf, df_xgb, df_lr], ignore_index=True)
df_melted = df_combined.melt(id_vars=["Classifier", "Input"], var_name="Metric", value_name="Value")

# Plot lines instead of bars
fig = px.line(
    df_melted,
    x="Metric",
    y="Value",
    color="Classifier",
    line_group="Classifier",
    markers=True,
    facet_col="Input",
    title="Classifier Performance Comparison (Line Plot)",
    labels={"Metric": "", "Value": ""}
)

fig.update_layout(height=600)
fig.write_image(os.path.join(plots_dir, "classifier_performance_line.svg"))

fig.show()


In [68]:
df_combined = pd.concat([df_lr,df_rf, df_xgb], ignore_index=True)[["Classifier", "Input", "AUC"]]

In [69]:
def create_spider_plot(input_category, filename):
    fig = go.Figure()
    classifiers = df_combined["Classifier"].unique()

    auc_values = []
    for classifier in classifiers:
        auc = df_combined[(df_combined["Classifier"] == classifier) & (df_combined["Input"] == input_category)]["AUC"].values
        auc_values.append(auc[0] if len(auc) > 0 else 0)

    auc_values.append(auc_values[0])
    angles = np.linspace(0, 2 * np.pi, len(classifiers), endpoint=False).tolist()
    angles += angles[:1]  

    fig.add_trace(go.Scatterpolar(
        r=auc_values,
        theta=classifiers.tolist() + [classifiers[0]],
        fill='toself',
        name=input_category
    ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title=f"AUC Spider Plot for {input_category}"
    )

    fig.write_image(os.path.join(plots_dir, filename))  # Save as PNG
    fig.show()

Spider plot for 3 PC input

In [70]:
error_metrics = ["Classifier", "Input", "MCC", "Precision", "Recall", "F1 Score", "Accuracy", "AUC"]
df_combined = pd.concat([df_rf[error_metrics],df_xgb[error_metrics], df_lr[error_metrics]], ignore_index=True)


In [47]:
df_3pc = df_combined[df_combined["Input"] == "3 PC"]
df_gls = df_combined[df_combined["Input"] == "3 PC + GLS"]
df_echo = df_combined[df_combined["Input"] == "3 PC + ECHO"]

In [48]:
def create_spider_plot(df, title, filename):
    fig = go.Figure()
    metrics = ["MCC", "Precision", "Recall", "F1 Score", "Accuracy", "AUC"]

    for classifier in df["Classifier"].unique():
        values = df[df["Classifier"] == classifier][metrics].values.flatten().tolist()
        values.append(values[0])
        angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
        angles += angles[:1]

        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=metrics + [metrics[0]],
            fill='toself',
            name=classifier
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title=title
    )

    fig.write_image(os.path.join(plots_dir, filename))  # Save as PNG
    fig.show()

In [49]:
create_spider_plot(df_3pc, "Error Metrics Spider Plot - 3 PC", "spider_plot_3pc.png")

In [50]:
create_spider_plot(df_gls, "Error Metrics Spider Plot - GLS", "spider_plot_gls.png")

In [51]:
create_spider_plot(df_echo, "Error Metrics Spider Plot - ECHO", "spider_plot_echo.png")